# Full Study — Block 1: External Label Efficiency + Pooling Sensitivity v1.0

This notebook is part of the frozen full-study protocol following:

- Phase-0A: `GO_ULTRASOUND_PRETRAINING_EXTERNAL_ADVANTAGE`
- Phase-0B: `GO_FULL_STUDY_TWO_USFM_REPLICATION`

No new paper direction is introduced here.

---

# RQ2 — Does the ultrasound-specific external-transfer advantage persist with fewer source labels?

Models:
1. URFM ViT-B/16 — ultrasound-specific
2. UltraFedFM ViT-B/16 — ultrasound-specific
3. ImageNet-MAE ViT-B/16 — architecture-matched generic baseline

Source-label fractions:
- 20%
- 50%
- 100%

Sources:
- BUS-BRA patient-level source split
- BrEaST case-level source split

External targets:
- reciprocal BrEaST / BUS-BRA
- BUSI image-level external cohort

## Replicates

Replicate seeds = 42, 43, 44.

For each replicate seed:
- pathology-stratified source-train subset is sampled once;
- the exact same source units are used for all three models;
- the same seed initializes the linear head.

Thus each replicate jointly captures:
- source-label subset sampling;
- head initialization.

Within each replicate, the 20% subset is nested inside the 50% subset.

The 100% condition reuses completed Phase-0A/B source-trained heads and results.

No target labels are used for model selection or hyperparameter tuning.

---

# RQ3-control — Does common pooling choice change the primary conclusion?

This is a reviewer-proof sensitivity control, not a new architecture experiment.

Primary Phase-0A/B used:
- common CLS-token features.

Sensitivity uses:
- common mean-patch token features.

For all three ViT-B/16 encoders:
- same checkpoint;
- same 224×224 image preprocessing;
- same encoder;
- same token normalization produced by the encoder;
- only pooling changes from CLS to arithmetic mean of patch tokens.

This is deliberately a **common mean-patch pooling control**. It is not claimed to reproduce each
model author's downstream head exactly.

Pooling sensitivity is run at 100% source labels only.

---

# Statistical unit and metrics

BUS-BRA:
- source and target patient-level;
- image features/scores averaged within patient.

BrEaST:
- case-level.

BUSI:
- image-level because reliable patient IDs are unavailable.

Primary metric:
- external AUROC.

Secondary:
- external AUPRC.

Uncertainty:
- 3000 pathology-stratified paired target bootstrap replicates.

All comparisons use the same target resamples for paired models.

---

# Frozen RQ2 interpretation

For each label fraction and each ultrasound FM, report:

\[
\Delta_{FM-MAE}(f, s, t)
=
AUC_{FM}(f,s,t)-AUC_{MAE}(f,s,t)
\]

Primary descriptive summaries:
1. median external FM−MAE effect across four scenarios;
2. number of positive scenarios;
3. number with paired-bootstrap `P(FM > MAE) >= .90`.

External-transfer advantage is called **retained at fraction f** for a given USFM only if:
- >=3/4 external scenarios have positive point effect; and
- median external FM−MAE effect >= +0.02 AUROC.

This is an interpretation rule, not a gate for adding rescue experiments.

We also report low-label degradation within each model:

\[
Drop_m(f)=AUC_m(100\%)-AUC_m(f)
\]

using matched scenario/replicate aggregation.

Permitted claim:
- whether ultrasound-specific advantage persists, attenuates, or disappears as source labels decrease.

Not permitted:
- universal few-shot superiority beyond these source-target cohorts.

---

# Frozen pooling-control interpretation

For each USFM, compare the FM−MAE external effect under CLS vs mean-patch pooling.

The original external-transfer conclusion is considered **pooling-stable** for a USFM if under
mean-patch pooling:
- >=3/4 scenario effects remain positive; and
- median FM−MAE effect remains >= +0.02 AUROC.

If this fails:
- report pooling dependence explicitly;
- do not hide the sensitivity;
- do not add another pooling rule to rescue the result.

---

# Stop rules for Block 1

This is a confirmatory characterization block, not another paper-existence kill-test.

However:
- if BOTH URFM and UltraFedFM fail the pooling-stability rule, full fine-tuning is not launched
  until the manuscript claim is narrowed;
- no third USFM, no PEFT, no new readout family, and no target tuning are allowed as rescue.

Full fine-tuning remains a later, separately frozen block.


In [ ]:
!pip -q install timm==1.0.28 huggingface_hub kagglehub pandas scikit-learn pillow tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, gc, json, random, hashlib
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import timm
import kagglehub
from huggingface_hub import hf_hub_download, login, whoami
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/SPATIOSPECTRAL_PEFT_PILOT')

PHASEA = ROOT/'fm_external_transfer_phase0_v1_0'
PHASEA_OUT = PHASEA/'outputs'
PHASEA_PROG = PHASEA/'progress'
PHASEA_CKPT = PHASEA/'checkpoints'

PHASEB = ROOT/'ultrafedfm_phase0b_v1_0'
PHASEB_OUT = PHASEB/'outputs'
PHASEB_PROG = PHASEB/'progress'
PHASEB_CKPT = PHASEB/'checkpoints'

PACKA = ROOT/'reviewer_proof_pack_A_v1_0'
PACKAO = PACKA/'outputs'
D1 = ROOT/'linear_readout_control_v1_0'
D1O = D1/'outputs'

URFM_DIR = ROOT/'external_models'/'URFM'
URFM_FILENAME = 'mae_vit_base_patch16_dec768d8b_all_biomedclip_1199.pth'
URFM_LOCAL = URFM_DIR/URFM_FILENAME
URFM_REPO_ID = 'QingboKang/URFM'

ULTRAFED_LOCAL = (
    ROOT/'external_models'/'UltraFedFM'/
    'official_pretrained_ultrafedfm_checkpoint.pth'
)

EXP = ROOT/'fullstudy_block1_label_pool_v1_0'
OUT = EXP/'outputs'
PROG = EXP/'progress'
CKPT = EXP/'checkpoints'
for p in [EXP,OUT,PROG,CKPT]:
    p.mkdir(parents=True,exist_ok=True)

MODELS = ['URFM','UltraFedFM','MAE']
USFMS = ['URFM','UltraFedFM']
FRACTIONS = [0.20,0.50,1.00]
REPLICATE_SEEDS = [42,43,44]

HEAD_BATCH = 16
HEAD_EPOCHS = 30
HEAD_LR = 1e-4
HEAD_WD = 1e-2

FEATURE_BATCH = 8
BOOTSTRAP_REPS = 3000
BOOTSTRAP_SEED = 97001

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:",DEVICE)
if DEVICE.type=='cuda':
    print(torch.cuda.get_device_name(0))

CONFIG = {
    'experiment':'Full Study Block 1: label efficiency + pooling sensitivity v1.0',
    'models':MODELS,
    'usfms':USFMS,
    'fractions':FRACTIONS,
    'replicate_seeds':REPLICATE_SEEDS,
    'replicate_definition':'same seed controls stratified subset sampling and head initialization',
    'subset_nesting':'20% nested within 50% within each replicate',
    'pooling':['CLS','MEAN_PATCH'],
    'pooling_sensitivity_fraction':1.0,
    'head_epochs':HEAD_EPOCHS,
    'bootstrap_reps':BOOTSTRAP_REPS,
    'target_tuning':False,
    'rescue_models_allowed':False,
}
(EXP/'FROZEN_BLOCK1_CONFIG.json').write_text(
    json.dumps(CONFIG,indent=2),encoding='utf-8'
)


## 1. Utilities

In [ ]:
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32).view(3,1,1)
IMAGENET_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32).view(3,1,1)

def seed_everything(seed):
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def atomic_csv(df,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False)
    os.replace(tmp,path)

def atomic_json(obj,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2),encoding='utf-8')
    os.replace(tmp,path)

def canonical_patient_key(v):
    s=str(v).strip()
    if re.fullmatch(r'\d+(?:\.0+)?',s):
        return str(int(float(s)))
    return s.lower()

def all_exist(values):
    return bool(np.all([Path(str(x)).exists() for x in values]))

def pil_tensor(img):
    img=img.convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
    x=torch.from_numpy(np.asarray(img,dtype=np.float32)/255.0).permute(2,0,1)
    return (x-IMAGENET_MEAN)/IMAGENET_STD

def auc_ap(y,s):
    return float(roc_auc_score(y,s)),float(average_precision_score(y,s))

def malignant_margin(logits):
    return logits[:,1]-logits[:,0]

def weights_for(y):
    c=np.bincount(np.asarray(y,dtype=int),minlength=2)
    w=c.sum()/(2*np.maximum(c,1))
    w=w/w.mean()
    return torch.tensor(w,dtype=torch.float32)

def sha256(path,block=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()


## 2. Load the same manifests and exact source splits

In [ ]:
busi_candidates=[
    PACKAO/'BUSI_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSI_LINEAR_CONTROL_MANIFEST.csv',
]
busi_path=next((p for p in busi_candidates if p.exists()),None)

busbra_candidates=[
    PACKAO/'BUSBRA_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSBRA_EXTERNAL_MANIFEST.csv',
]
busbra_path=next((p for p in busbra_candidates if p.exists()),None)

breast_path=PACKAO/'BREAST_LESION_MANIFEST.csv'

if busi_path is None or busbra_path is None or not breast_path.exists():
    raise FileNotFoundError("Required project manifests are missing.")

busi=pd.read_csv(busi_path)
busbra=pd.read_csv(busbra_path)
breast=pd.read_csv(breast_path)

if not all_exist(busi.image_path):
    print("Remapping stale BUSI paths with KaggleHub.")
    root=Path(kagglehub.dataset_download('aryashah2k/breast-ultrasound-images-dataset'))
    idx={}
    for p in root.rglob('*.png'):
        if '_mask' in p.stem.lower(): continue
        idx[(p.parent.name.strip().lower(),p.stem)] = str(p)

    resolved=[]
    missing=[]
    for _,r in busi.iterrows():
        key=(str(r.label_text).strip().lower(),str(r.image_id).strip())
        p=idx.get(key)
        if p is None: missing.append(key)
        resolved.append(p)
    if missing:
        raise RuntimeError(f"BUSI remap failed: {missing[:10]}")
    busi=busi.copy()
    busi['image_path']=resolved

if not all_exist(busbra.image_path):
    candidates=[
        ROOT/'data'/'BUSBRA_OFFICIAL'/'BUSBRA'/'Images',
        ROOT/'data'/'BUS-BRA'/'Images',
    ]
    image_dir=next((p for p in candidates if p.exists()),None)
    if image_dir is None:
        raise FileNotFoundError("BUS-BRA raw image directory not found.")
    remapped=[str(image_dir/f'{str(iid)}.png') for iid in busbra.image_id]
    if not all_exist(remapped):
        raise RuntimeError("BUS-BRA path remap failed.")
    busbra=busbra.copy()
    busbra['image_path']=remapped

if not all_exist(breast.image_path):
    raise FileNotFoundError("BrEaST extracted image paths are stale.")

busi_all=pd.concat([
    busi[busi.split=='train'].reset_index(drop=True),
    busi[busi.split=='val'].reset_index(drop=True)
],ignore_index=True)

busbra['patient_key']=busbra.patient_key.map(canonical_patient_key)
breast['patient_key']=breast.patient_key.astype(str)

assert len(busi_all)==647
assert len(busbra)==1875 and busbra.patient_key.nunique()==1064
assert len(breast)==252 and breast.patient_key.nunique()==252

split_paths={
    'BUSBRA':PHASEA_OUT/'BUSBRA_PHASE0_SOURCE_SPLIT.csv',
    'BREAST':PHASEA_OUT/'BREAST_PHASE0_SOURCE_SPLIT.csv',
}
saved_splits={k:pd.read_csv(v) for k,v in split_paths.items()}

print("PASS manifests and frozen source splits.")


## 3. Reuse completed CLS feature caches

No CLS backbone inference is repeated.


In [ ]:
cls_paths={
    ('URFM','BUSI'):PHASEA_PROG/'URFM_224_BUSI_features.npy',
    ('URFM','BUSBRA'):PHASEA_PROG/'URFM_224_BUSBRA_features.npy',
    ('URFM','BREAST'):PHASEA_PROG/'URFM_224_BREAST_features.npy',

    ('MAE','BUSI'):PHASEA_PROG/'ImageNet_MAE_224_BUSI_features.npy',
    ('MAE','BUSBRA'):PHASEA_PROG/'ImageNet_MAE_224_BUSBRA_features.npy',
    ('MAE','BREAST'):PHASEA_PROG/'ImageNet_MAE_224_BREAST_features.npy',

    ('UltraFedFM','BUSI'):PHASEB_PROG/'UltraFedFM_224_BUSI_features.npy',
    ('UltraFedFM','BUSBRA'):PHASEB_PROG/'UltraFedFM_224_BUSBRA_features.npy',
    ('UltraFedFM','BREAST'):PHASEB_PROG/'UltraFedFM_224_BREAST_features.npy',
}

missing={f'{m}/{d}':str(p) for (m,d),p in cls_paths.items() if not p.exists()}
if missing:
    raise FileNotFoundError(
        "Completed CLS feature cache(s) missing; do not silently recompute:\n"
        +json.dumps(missing,indent=2)
    )

features_cls={
    (m,d):np.asarray(np.load(p,mmap_mode='r'),dtype=np.float32)
    for (m,d),p in cls_paths.items()
}

expected={'BUSI':647,'BUSBRA':1875,'BREAST':252}
for (m,d),X in features_cls.items():
    assert X.shape==(expected[d],768),(m,d,X.shape)

print("PASS all nine CLS feature caches.")


## 4. Build source statistical units

In [ ]:
def patient_mean_features(X,manifest):
    groups=manifest.groupby(['patient_key','y'],sort=True).indices
    rows=[]
    vecs=[]
    for (pid,y),idx in groups.items():
        idx=np.asarray(idx,dtype=int)
        rows.append({'patient_key':str(pid),'y':int(y),'n_images':len(idx)})
        vecs.append(np.asarray(X[idx],dtype=np.float32).mean(axis=0))
    return pd.DataFrame(rows),np.stack(vecs).astype(np.float32)

def build_source_units(feature_dict):
    out={}
    for model in MODELS:
        meta_bra,X_bra=patient_mean_features(feature_dict[(model,'BUSBRA')],busbra)
        out[(model,'BUSBRA')]={'meta':meta_bra,'X':X_bra,'y':meta_bra.y.to_numpy(int)}

        meta_br=breast[['patient_key','y']].copy().reset_index(drop=True)
        out[(model,'BREAST')]={'meta':meta_br,'X':feature_dict[(model,'BREAST')],'y':meta_br.y.to_numpy(int)}
    return out

units_cls=build_source_units(features_cls)

splits={}
for source in ['BUSBRA','BREAST']:
    ref=units_cls[('URFM',source)]['meta'].copy()
    saved=saved_splits[source].copy()
    saved['patient_key']=saved.patient_key.astype(str)
    smap=dict(zip(saved.patient_key,saved.source_split))
    labels=ref.patient_key.astype(str).map(smap)
    if labels.isna().any():
        raise RuntimeError(f"{source}: exact Phase-0A split could not be recovered.")
    tr=np.flatnonzero(labels.to_numpy()=='train')
    va=np.flatnonzero(labels.to_numpy()=='val')
    splits[source]=(tr,va)

    for model in MODELS[1:]:
        cur=units_cls[(model,source)]['meta']
        assert np.array_equal(ref.patient_key.astype(str).to_numpy(),cur.patient_key.astype(str).to_numpy())
        assert np.array_equal(ref.y.to_numpy(int),cur.y.to_numpy(int))

    print(source,"train",len(tr),"val",len(va))


## 5. Freeze nested 20% / 50% source-label subsets

Subsets are sampled within the original source-training partition only.

For each replicate:
- shuffle benign training indices;
- shuffle malignant training indices;
- take the first class-wise fraction;
- therefore 20% is nested within 50%.

The exact subset manifest is saved before training.


In [ ]:
def nested_subset_indices(y,train_idx,seed,fraction):
    y=np.asarray(y,dtype=int)
    train_idx=np.asarray(train_idx,dtype=int)
    rng=np.random.default_rng(seed)

    selected=[]
    for cls in [0,1]:
        idx=train_idx[y[train_idx]==cls].copy()
        rng.shuffle(idx)
        n=max(1,int(round(len(idx)*fraction)))
        if fraction>=1.0:
            n=len(idx)
        selected.extend(idx[:n].tolist())
    return np.asarray(sorted(selected),dtype=int)

subset_rows=[]
subset_idx={}

for source in ['BUSBRA','BREAST']:
    y=units_cls[('URFM',source)]['y']
    meta=units_cls[('URFM',source)]['meta']
    tr,_=splits[source]

    for seed in REPLICATE_SEEDS:
        s20=nested_subset_indices(y,tr,seed,0.20)
        s50=nested_subset_indices(y,tr,seed,0.50)
        s100=np.asarray(sorted(tr),dtype=int)

        if not set(s20).issubset(set(s50)):
            raise RuntimeError(f"{source}/seed{seed}: 20% is not nested within 50%.")

        for frac,idx in [(0.20,s20),(0.50,s50),(1.00,s100)]:
            subset_idx[(source,seed,frac)]=idx
            for i in idx:
                subset_rows.append({
                    'source':source,
                    'replicate_seed':seed,
                    'fraction':frac,
                    'unit_index':int(i),
                    'patient_key':str(meta.iloc[i].patient_key),
                    'y':int(meta.iloc[i].y),
                })

subset_manifest=pd.DataFrame(subset_rows)
atomic_csv(subset_manifest,OUT/'FROZEN_LABEL_EFFICIENCY_SUBSETS.csv')

summary=(
    subset_manifest.groupby(['source','replicate_seed','fraction','y'])
    .size().rename('n').reset_index()
)
display(summary)


## 6. Linear-head training utility

In [ ]:
class FeatDS(Dataset):
    def __init__(self,X,y):
        self.X=np.asarray(X,dtype=np.float32)
        self.y=np.asarray(y,dtype=np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return torch.from_numpy(self.X[i]),torch.tensor(int(self.y[i]),dtype=torch.long)

@torch.no_grad()
def head_scores(head,X):
    head.eval()
    loader=DataLoader(
        torch.from_numpy(np.asarray(X,dtype=np.float32)),
        batch_size=256,shuffle=False,num_workers=0
    )
    out=[]
    for x in loader:
        out.extend(malignant_margin(head(x)).numpy().tolist())
    return np.asarray(out,dtype=float)

def train_head(X,y,train_idx,val_idx,seed,tag):
    best_path=CKPT/f'{tag}_best.pt'
    hist_path=OUT/f'{tag}_history.csv'
    status_path=OUT/f'{tag}_status.json'

    if best_path.exists() and status_path.exists():
        st=json.loads(status_path.read_text(encoding='utf-8'))
        if st.get('complete_30_epochs',False):
            h=nn.Linear(768,2)
            ck=torch.load(best_path,map_location='cpu',weights_only=False)
            h.load_state_dict(ck['state_dict'])
            h.eval()
            return h,st

    seed_everything(seed)

    gen=torch.Generator()
    gen.manual_seed(seed)

    loader=DataLoader(
        FeatDS(X[train_idx],y[train_idx]),
        batch_size=HEAD_BATCH,
        shuffle=True,
        num_workers=0,
        generator=gen
    )

    h=nn.Linear(768,2)
    opt=torch.optim.AdamW(h.parameters(),lr=HEAD_LR,weight_decay=HEAD_WD)
    cw=weights_for(y[train_idx])

    best_auc=-np.inf
    best_epoch=None
    hist=[]

    for epoch in range(1,HEAD_EPOCHS+1):
        h.train()
        losses=[]
        for f,yy in loader:
            opt.zero_grad(set_to_none=True)
            logits=h(f)
            loss=F.cross_entropy(logits,yy,weight=cw)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        s=head_scores(h,X[val_idx])
        auc,ap=auc_ap(y[val_idx],s)

        hist.append({
            'epoch':epoch,
            'train_loss':float(np.mean(losses)),
            'source_val_auc':auc,
            'source_val_auprc':ap,
        })

        if auc>best_auc:
            best_auc=auc
            best_epoch=epoch
            torch.save({
                'state_dict':{k:v.detach().cpu().clone() for k,v in h.state_dict().items()},
                'epoch':epoch,
                'best_source_val_auc':best_auc,
            },best_path)

    atomic_csv(pd.DataFrame(hist),hist_path)

    ck=torch.load(best_path,map_location='cpu',weights_only=False)
    h.load_state_dict(ck['state_dict'])
    h.eval()

    st={
        'complete_30_epochs':True,
        'best_epoch':int(best_epoch),
        'best_source_val_auc':float(best_auc),
        'trainable_params':1538,
    }
    atomic_json(st,status_path)
    return h,st


## 7. RQ2 label-efficiency heads

20% and 50% are newly trained.

100% uses the completed Phase-0A/B heads rather than repeating identical training.


In [ ]:
def load_existing_cls_head(model,source,seed):
    if model=='URFM':
        p=PHASEA_CKPT/f'URFM_224_{source}_seed{seed}_best.pt'
    elif model=='MAE':
        p=PHASEA_CKPT/f'ImageNet_MAE_224_{source}_seed{seed}_best.pt'
    elif model=='UltraFedFM':
        p=PHASEB_CKPT/f'UltraFedFM_224_{source}_seed{seed}_best.pt'
    else:
        raise ValueError(model)

    if not p.exists():
        raise FileNotFoundError(p)

    ck=torch.load(p,map_location='cpu',weights_only=False)
    h=nn.Linear(768,2)
    h.load_state_dict(ck['state_dict'])
    h.eval()
    return h

heads_label={}
status_rows=[]

for model in MODELS:
    for source in ['BUSBRA','BREAST']:
        X=units_cls[(model,source)]['X']
        y=units_cls[(model,source)]['y']
        _,va=splits[source]

        for seed in REPLICATE_SEEDS:
            for frac in FRACTIONS:
                if frac==1.00:
                    h=load_existing_cls_head(model,source,seed)
                    st={
                        'model':model,'source':source,'replicate_seed':seed,'fraction':frac,
                        'status':'reused_completed_phase0_head','complete_30_epochs':True
                    }
                else:
                    tr_sub=subset_idx[(source,seed,frac)]
                    tag=f'LABEL_{model}_{source}_f{int(frac*100)}_seed{seed}'
                    h,st0=train_head(X,y,tr_sub,va,seed,tag)
                    st={
                        'model':model,'source':source,'replicate_seed':seed,'fraction':frac,
                        'status':'new_block1_head',**st0
                    }

                heads_label[(model,source,seed,frac)]=h
                status_rows.append(st)

status_label=pd.DataFrame(status_rows)
atomic_csv(status_label,OUT/'LABEL_EFFICIENCY_HEAD_STATUS.csv')
display(status_label)


## 8. RQ2 external evaluation

In [ ]:
def busbra_patient_scores(img_scores):
    t=busbra[['patient_key','y']].copy().reset_index(drop=True)
    t['score']=np.asarray(img_scores,dtype=float)
    return (
        t.groupby(['patient_key','y'],as_index=False)
        .agg(score=('score','mean'))
        .sort_values('patient_key').reset_index(drop=True)
    )

scenarios=[
    ('BUSBRA','BREAST'),
    ('BUSBRA','BUSI'),
    ('BREAST','BUSBRA'),
    ('BREAST','BUSI'),
]

def target_scores_from_features(head,model,target,feature_dict):
    if target=='BUSBRA':
        raw=head_scores(head,feature_dict[(model,'BUSBRA')])
        p=busbra_patient_scores(raw)
        return (
            p.patient_key.astype(str).to_numpy(),
            p.y.to_numpy(int),
            p.score.to_numpy(float)
        )
    if target=='BREAST':
        return (
            breast.patient_key.astype(str).to_numpy(),
            breast.y.to_numpy(int),
            head_scores(head,feature_dict[(model,'BREAST')])
        )
    if target=='BUSI':
        return (
            busi_all.image_id.astype(str).to_numpy(),
            busi_all.y.to_numpy(int),
            head_scores(head,feature_dict[(model,'BUSI')])
        )
    raise ValueError(target)

label_scores={}
label_rows=[]

for source,target in scenarios:
    for model in MODELS:
        for seed in REPLICATE_SEEDS:
            for frac in FRACTIONS:
                h=heads_label[(model,source,seed,frac)]
                ids,y,s=target_scores_from_features(h,model,target,features_cls)
                label_scores[(model,source,target,seed,frac)]={'ids':ids,'y':y,'score':s}
                auc,ap=auc_ap(y,s)
                label_rows.append({
                    'model':model,'source':source,'target':target,
                    'replicate_seed':seed,'fraction':frac,'n':len(y),
                    'auc':auc,'auprc':ap
                })

label_external=pd.DataFrame(label_rows)
atomic_csv(label_external,OUT/'LABEL_EFFICIENCY_EXTERNAL_RESULTS.csv')
display(label_external.round(4))


## 9. RQ2 paired bootstrap summaries

In [ ]:
def paired_effect_bootstrap(score_dict,model_a,model_b,source,target,frac,seed_base):
    # Point effect is mean over replicate seeds.
    ref=score_dict[(model_a,source,target,REPLICATE_SEEDS[0],frac)]
    ids=ref['ids']; y=ref['y']

    for model in [model_a,model_b]:
        for seed in REPLICATE_SEEDS:
            cur=score_dict[(model,source,target,seed,frac)]
            assert np.array_equal(ids,cur['ids'])
            assert np.array_equal(y,cur['y'])

    point_d=[]
    for seed in REPLICATE_SEEDS:
        a=score_dict[(model_a,source,target,seed,frac)]['score']
        b=score_dict[(model_b,source,target,seed,frac)]['score']
        point_d.append(roc_auc_score(y,a)-roc_auc_score(y,b))
    point=float(np.mean(point_d))

    i0=np.flatnonzero(y==0)
    i1=np.flatnonzero(y==1)
    rng=np.random.default_rng(seed_base)
    boots=[]

    for _ in range(BOOTSTRAP_REPS):
        idx=np.concatenate([
            rng.choice(i0,size=len(i0),replace=True),
            rng.choice(i1,size=len(i1),replace=True),
        ])
        yy=y[idx]
        ds=[]
        for seed in REPLICATE_SEEDS:
            a=score_dict[(model_a,source,target,seed,frac)]['score'][idx]
            b=score_dict[(model_b,source,target,seed,frac)]['score'][idx]
            ds.append(roc_auc_score(yy,a)-roc_auc_score(yy,b))
        boots.append(float(np.mean(ds)))

    boots=np.asarray(boots)
    return {
        'point':point,
        'seed42_delta':float(point_d[0]),
        'seed43_delta':float(point_d[1]),
        'seed44_delta':float(point_d[2]),
        'bootstrap_median':float(np.median(boots)),
        'ci_lo':float(np.quantile(boots,.025)),
        'ci_hi':float(np.quantile(boots,.975)),
        'p_a_better':float(np.mean(boots>0)),
    },boots

rq2_rows=[]
rq2_boot=[]

counter=0
for usfm in USFMS:
    for frac in FRACTIONS:
        for source,target in scenarios:
            eff,boots=paired_effect_bootstrap(
                label_scores,usfm,'MAE',source,target,frac,
                BOOTSTRAP_SEED+counter
            )
            counter+=1
            rq2_rows.append({
                'usfm':usfm,'fraction':frac,'source':source,'target':target,
                **eff
            })
            rq2_boot.append(pd.DataFrame({
                'usfm':usfm,'fraction':frac,'source':source,'target':target,
                'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
                'fm_minus_mae_auc':boots
            }))

rq2=pd.DataFrame(rq2_rows)
rq2_boot=pd.concat(rq2_boot,ignore_index=True)
atomic_csv(rq2,OUT/'RQ2_USFM_MINUS_MAE_EXTERNAL_EFFECTS.csv')
atomic_csv(rq2_boot,OUT/'RQ2_BOOTSTRAP_3000.csv')

retention=[]
for usfm in USFMS:
    for frac in FRACTIONS:
        sub=rq2[(rq2.usfm==usfm)&(rq2.fraction==frac)]
        positive=int((sub.point>0).sum())
        med=float(sub.point.median())
        supported=int((sub.p_a_better>=.90).sum())
        retention.append({
            'usfm':usfm,'fraction':frac,
            'positive_scenarios':positive,
            'median_fm_minus_mae_auc':med,
            'bootstrap_supported_scenarios':supported,
            'retained_external_advantage':bool(positive>=3 and med>=.02)
        })

retention=pd.DataFrame(retention)
atomic_csv(retention,OUT/'RQ2_LABEL_EFFICIENCY_RETENTION_SUMMARY.csv')

display(rq2.round(4))
display(retention.round(4))


## 10. Load encoders for common mean-patch feature extraction

If the mean-patch caches are already complete, GPU/model loading may be skipped manually.

URFM:
- existing official gated checkpoint on Drive is preferred;
- if missing, the notebook requests Hugging Face authentication.

UltraFedFM:
- requires the already downloaded official pretrained checkpoint on Drive.

MAE:
- official timm pretrained model.


In [ ]:
def resolve_urfm():
    if URFM_LOCAL.exists():
        return str(URFM_LOCAL)
    print("URFM checkpoint absent from Drive; gated Hugging Face authentication required.")
    login()
    p=hf_hub_download(repo_id=URFM_REPO_ID,filename=URFM_FILENAME)
    URFM_DIR.mkdir(parents=True,exist_ok=True)
    import shutil
    shutil.copy2(p,URFM_LOCAL)
    return str(URFM_LOCAL)

def filtered_load_vit_base(checkpoint_path,container_order=('model','state_dict')):
    ck=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
    if not isinstance(ck,dict):
        raise RuntimeError("Checkpoint is not a dict.")

    state=None
    container='raw'
    for k in container_order:
        if k in ck and isinstance(ck[k],dict):
            state=ck[k]; container=k; break
    if state is None:
        state=ck

    cleaned={}
    for k,v in state.items():
        nk=k
        for prefix in ['module.','model.','encoder.']:
            if nk.startswith(prefix):
                nk=nk[len(prefix):]
        cleaned[nk]=v

    model=timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        num_classes=0,
        global_pool='token'
    )
    target=model.state_dict()
    filt={}
    for k,v in cleaned.items():
        if k.startswith('decoder') or k=='mask_token' or k.startswith('head.'):
            continue
        if k in target and tuple(v.shape)==tuple(target[k].shape):
            filt[k]=v

    msg=model.load_state_dict(filt,strict=False)
    critical=[
        'patch_embed.proj.weight','cls_token','pos_embed',
        'blocks.0.attn.qkv.weight','blocks.11.attn.qkv.weight',
        'norm.weight','norm.bias'
    ]
    absent=[k for k in critical if k not in filt]
    if absent:
        raise RuntimeError(f"Critical encoder keys missing: {absent}")

    unexpected_missing=[k for k in msg.missing_keys if not k.startswith('head.')]
    if unexpected_missing:
        raise RuntimeError(f"Unexpected encoder missing keys: {unexpected_missing[:20]}")
    return model,{'container':container,'loaded_tensors':len(filt)}

def build_pool_model(model_name):
    if model_name=='URFM':
        p=resolve_urfm()
        model,audit=filtered_load_vit_base(p)
        audit['checkpoint']=p
    elif model_name=='UltraFedFM':
        if not ULTRAFED_LOCAL.exists():
            raise FileNotFoundError(ULTRAFED_LOCAL)
        model,audit=filtered_load_vit_base(str(ULTRAFED_LOCAL))
        audit['checkpoint']=str(ULTRAFED_LOCAL)
    elif model_name=='MAE':
        try:
            model=timm.create_model(
                'vit_base_patch16_224.mae',
                pretrained=True,num_classes=0,global_pool='token'
            )
        except Exception:
            model=timm.create_model(
                'hf_hub:timm/vit_base_patch16_224.mae',
                pretrained=True,num_classes=0,global_pool='token'
            )
        audit={'checkpoint':'timm/vit_base_patch16_224.mae'}
    else:
        raise ValueError(model_name)

    for p in model.parameters():
        p.requires_grad=False
    model=model.to(DEVICE).eval()

    with torch.no_grad():
        toks=model.forward_features(torch.zeros(1,3,224,224,device=DEVICE))
    if toks.ndim!=3 or toks.shape[-1]!=768 or toks.shape[1] < 2:
        raise RuntimeError(f"{model_name}: unexpected token output {tuple(toks.shape)}")

    audit['forward_features_shape']=list(toks.shape)
    atomic_json(audit,OUT/f'{model_name}_MEAN_PATCH_LOAD_AUDIT.json')
    return model


## 11. Interruption-safe common mean-patch feature extraction

In [ ]:
class ImageRows(Dataset):
    def __init__(self,rows):
        self.rows=rows.reset_index(drop=True)
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows.iloc[i]
        img=Image.open(r.image_path).convert('RGB')
        return {'x':pil_tensor(img),'idx':torch.tensor(i,dtype=torch.long)}

def open_mm(path,dtype,shape):
    path=Path(path)
    if not path.exists():
        mm=np.lib.format.open_memmap(path,mode='w+',dtype=dtype,shape=shape)
        mm[:]=0
        mm.flush()
        del mm
    mm=np.load(path,mmap_mode='r+')
    if mm.shape!=shape:
        raise RuntimeError(f"{path.name}: {mm.shape} != {shape}")
    return mm

@torch.no_grad()
def extract_mean_patch(model,model_name,rows,dataset):
    ds=ImageRows(rows)
    n=len(ds)
    fp=PROG/f'{model_name}_MEAN_PATCH_{dataset}_features.npy'
    dp=PROG/f'{model_name}_MEAN_PATCH_{dataset}_done.npy'
    feat=open_mm(fp,np.float32,(n,768))
    done=open_mm(dp,np.bool_,(n,))

    remaining=np.flatnonzero(~np.asarray(done,dtype=bool)).tolist()
    print(model_name,dataset,":",n-len(remaining),"/",n,"complete")

    if remaining:
        if DEVICE.type!='cuda':
            raise RuntimeError("GPU required for unfinished mean-patch extraction.")
        loader=DataLoader(
            Subset(ds,remaining),batch_size=FEATURE_BATCH,
            shuffle=False,num_workers=0,pin_memory=True
        )
        model.eval()
        for b in tqdm(loader,desc=f'{model_name}-MEAN-{dataset}'):
            idx=b['idx'].numpy().astype(int)
            x=b['x'].to(DEVICE,non_blocking=True)
            toks=model.forward_features(x)
            z=toks[:,1:,:].mean(dim=1)
            feat[idx]=z.detach().cpu().numpy().astype(np.float32)
            done[idx]=True
            feat.flush(); done.flush()

    if not bool(np.asarray(done).all()):
        raise RuntimeError(f"Incomplete mean-patch cache: {model_name}/{dataset}")

    out=np.asarray(feat,dtype=np.float32).copy()
    del feat,done
    return out

dataset_rows={'BUSI':busi_all,'BUSBRA':busbra,'BREAST':breast}
features_mean={}

for model_name in MODELS:
    m=build_pool_model(model_name)
    for dataset,rows in dataset_rows.items():
        features_mean[(model_name,dataset)]=extract_mean_patch(
            m,model_name,rows,dataset
        )
    m.to('cpu')
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("PASS all mean-patch caches.")


## 12. Train 100% mean-patch heads and evaluate external transfer

In [ ]:
units_mean=build_source_units(features_mean)

# Check same unit identities.
for source in ['BUSBRA','BREAST']:
    ref=units_cls[('URFM',source)]['meta']
    for model in MODELS:
        cur=units_mean[(model,source)]['meta']
        assert np.array_equal(ref.patient_key.astype(str).to_numpy(),cur.patient_key.astype(str).to_numpy())
        assert np.array_equal(ref.y.to_numpy(int),cur.y.to_numpy(int))

heads_mean={}
mean_status=[]

for model in MODELS:
    for source in ['BUSBRA','BREAST']:
        X=units_mean[(model,source)]['X']
        y=units_mean[(model,source)]['y']
        tr,va=splits[source]

        for seed in REPLICATE_SEEDS:
            tag=f'MEAN_{model}_{source}_f100_seed{seed}'
            h,st0=train_head(X,y,tr,va,seed,tag)
            heads_mean[(model,source,seed)]=h
            mean_status.append({
                'model':model,'source':source,'seed':seed,**st0
            })

mean_status=pd.DataFrame(mean_status)
atomic_csv(mean_status,OUT/'MEAN_PATCH_HEAD_STATUS.csv')

mean_scores={}
mean_rows=[]

for source,target in scenarios:
    for model in MODELS:
        for seed in REPLICATE_SEEDS:
            h=heads_mean[(model,source,seed)]
            ids,y,s=target_scores_from_features(h,model,target,features_mean)
            mean_scores[(model,source,target,seed,1.0)]={'ids':ids,'y':y,'score':s}
            auc,ap=auc_ap(y,s)
            mean_rows.append({
                'model':model,'source':source,'target':target,
                'seed':seed,'auc':auc,'auprc':ap,'n':len(y)
            })

mean_external=pd.DataFrame(mean_rows)
atomic_csv(mean_external,OUT/'MEAN_PATCH_EXTERNAL_RESULTS.csv')
display(mean_external.round(4))


## 13. Pooling sensitivity: USFM−MAE effects under mean-patch vs CLS

In [ ]:
pool_rows=[]
pool_boot=[]
counter=1000

for usfm in USFMS:
    for source,target in scenarios:
        eff_mean,boots=paired_effect_bootstrap(
            mean_scores,usfm,'MAE',source,target,1.0,
            BOOTSTRAP_SEED+counter
        )
        counter+=1

        # CLS 100% effect already computed in RQ2.
        rcls=rq2[
            (rq2.usfm==usfm)&(rq2.fraction==1.0)&
            (rq2.source==source)&(rq2.target==target)
        ].iloc[0]

        pool_rows.append({
            'usfm':usfm,'source':source,'target':target,
            'cls_fm_minus_mae_auc':float(rcls.point),
            'mean_patch_fm_minus_mae_auc':float(eff_mean['point']),
            'pooling_change_in_effect':float(eff_mean['point']-rcls.point),
            'mean_patch_ci_lo':float(eff_mean['ci_lo']),
            'mean_patch_ci_hi':float(eff_mean['ci_hi']),
            'mean_patch_p_fm_better':float(eff_mean['p_a_better']),
        })
        pool_boot.append(pd.DataFrame({
            'usfm':usfm,'source':source,'target':target,
            'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
            'mean_patch_fm_minus_mae_auc':boots
        }))

pool=pd.DataFrame(pool_rows)
pool_boot=pd.concat(pool_boot,ignore_index=True)

pool_summary=[]
for usfm in USFMS:
    sub=pool[pool.usfm==usfm]
    pos=int((sub.mean_patch_fm_minus_mae_auc>0).sum())
    med=float(sub.mean_patch_fm_minus_mae_auc.median())
    pool_summary.append({
        'usfm':usfm,
        'positive_mean_patch_scenarios':pos,
        'median_mean_patch_fm_minus_mae_auc':med,
        'pooling_stable':bool(pos>=3 and med>=.02)
    })

pool_summary=pd.DataFrame(pool_summary)

atomic_csv(pool,OUT/'RQ3_POOLING_SENSITIVITY_EFFECTS.csv')
atomic_csv(pool_boot,OUT/'RQ3_POOLING_BOOTSTRAP_3000.csv')
atomic_csv(pool_summary,OUT/'RQ3_POOLING_STABILITY_SUMMARY.csv')

display(pool.round(4))
display(pool_summary.round(4))


## 14. Block-1 decision summary and package

In [ ]:
both_pooling_stable=bool(pool_summary.pooling_stable.all())

decision={
    'block':'Full Study Block 1',
    'rq2_label_efficiency':{
        'retention_summary':retention.to_dict(orient='records')
    },
    'rq3_pooling_control':{
        'summary':pool_summary.to_dict(orient='records'),
        'both_usfms_pooling_stable':both_pooling_stable
    },
    'next_step':(
        'PROCEED_TO_FROZEN_FULL_FINETUNING_BLOCK'
        if both_pooling_stable
        else 'NARROW_CLAIM_BEFORE_ANY_FULL_FINETUNING'
    ),
    'rescue_rule':(
        'No third USFM, PEFT, target tuning, or alternative pooling may be added to rescue a failed pooling control.'
    )
}
atomic_json(decision,OUT/'BLOCK1_DECISION_SUMMARY.json')
print(json.dumps(decision,indent=2))

result_zip=ROOT/'FULLSTUDY_BLOCK1_LABEL_POOL_RESULTS.zip'

files=[
    EXP/'FROZEN_BLOCK1_CONFIG.json',
    OUT/'FROZEN_LABEL_EFFICIENCY_SUBSETS.csv',
    OUT/'LABEL_EFFICIENCY_HEAD_STATUS.csv',
    OUT/'LABEL_EFFICIENCY_EXTERNAL_RESULTS.csv',
    OUT/'RQ2_USFM_MINUS_MAE_EXTERNAL_EFFECTS.csv',
    OUT/'RQ2_BOOTSTRAP_3000.csv',
    OUT/'RQ2_LABEL_EFFICIENCY_RETENTION_SUMMARY.csv',
    OUT/'MEAN_PATCH_HEAD_STATUS.csv',
    OUT/'MEAN_PATCH_EXTERNAL_RESULTS.csv',
    OUT/'RQ3_POOLING_SENSITIVITY_EFFECTS.csv',
    OUT/'RQ3_POOLING_BOOTSTRAP_3000.csv',
    OUT/'RQ3_POOLING_STABILITY_SUMMARY.csv',
    OUT/'BLOCK1_DECISION_SUMMARY.json',
]
for model in MODELS:
    files.append(OUT/f'{model}_MEAN_PATCH_LOAD_AUDIT.json')

with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files:
        if not Path(p).exists():
            raise FileNotFoundError(p)
        z.write(p,arcname=Path(p).name)

print("Created:",result_zip)
